In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint
from tqdm import tqdm

In [ ]:
from src.kg_model import KnowledgeGraphModel, KnowledgeGraphModelConfig
from src.db_drivers.vector_driver import EmbedderModelConfig
from src.pipelines.memorize import MemPipelineConfig, MemPipeline, LLMUpdatorConfig

from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig, WeakKGReasoner
from src.pipelines.qa.kg_reasoning.weak_reasoner import QALLMGeneratorConfig, QueryLLMParserConfig, KnowledgeComparatorConfig
from src.pipelines.qa.knowledge_retriever import KnowledgeRetrieverConfig

from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig, MediumKGReasoner
from src.pipelines.qa.kg_reasoning.medium_reasoner import AnswerGeneratorConfig, ClueAnswerGeneratorConfig, ClueAnswersSummarizerConfig, \
    ClueQueriesGeneratorConfig, EntitiesExtractorConfig, Entities2NodesMatcherConfig, SearchPlanEnhancerConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Пример настройки KGReasoner-стадии в рамках QA-пайплайна

1. Инициализация модели графа знаний

In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)

EMBEDDER_MODEL_PATH = '../../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['m-e5-base'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
kg_model = KnowledgeGraphModel(kg_config)

No sentence-transformers model found with name ../../../../models/intfloat/multilingual-e5-base. Creating a new one with mean pooling.


2. Инициализация Memorize-пайплайна

In [5]:
mem_config = MemPipelineConfig(
    updator_config=LLMUpdatorConfig(
        delete_obsolete_info=False # Выключаем механизм по поиску/удалению устревших знаний в графе при добавлении новой информации
    )
)
mem_pipeline = MemPipeline(kg_model, mem_config)

In [6]:
kg_model.clear()

3. Добавление информации в граф знаний

In [6]:
TEXT_EXAMPLES = [
    "Sasha was walking along the highway.", 
    "Masha was walking along the highway.", 
    "The ship was sailing along the water canal.", 
    "The motorboat was sailing along the river."]

In [8]:
extracted_triplets = []
for example in tqdm(TEXT_EXAMPLES):
    tmp_extracted_triplets, _, _ = mem_pipeline.remember(example)
    extracted_triplets += tmp_extracted_triplets

100%|██████████| 4/4 [00:31<00:00,  7.86s/it]


In [7]:
kg_model.count_items(detailed=True)

{'graph_info': {'triplets': {'simple': 5,
   'hyper': 10,
   'episodic': 15,
   'time': 0},
  'nodes': {'object': 9, 'hyper': 5, 'episodic': 4, 'time': 0}},
 'embeddings_info': {'nodes': {'object': {'dense_nodes': 9, 'bm25_nodes': 9},
   'hyper': {'dense_nodes': 5, 'bm25_nodes': 5},
   'episodic': {'dense_nodes': 4, 'bm25_nodes': 4},
   'time': {'dense_nodes': 0, 'bm25_nodes': 0}},
  'triplets': {'dense_triplets': 14, 'bm25_triplets': 14}},
 'nodestree_info': None}

In [8]:
kg_model.check_consistency()

True

In [9]:
print("mem tgen cache:")
pprint(mem_pipeline.get_agent_tgen_stat())
print("mem casual cache:")
pprint(mem_pipeline.get_cache_stat())

mem tgen cache:
{'extractor': {'thesises_extraction_solver': None,
               'triplets_extraction_solver': None},
 'updator': {'replace_hyper_solver': None, 'replace_simple_solver': None}}
mem casual cache:
{'MemPipeline': None,
 'extractor': {'LLMExtractor': None,
               'thesises_extraction_solver': None,
               'triplets_extraction_solver': None},
 'updator': {'LLMUpdator': None,
             'replace_hyper_solver': None,
             'replace_simple_solver': None}}


4.1. Инициализация 'weak' KGReasoner-стадии

In [12]:
weak_reasoner_config = WeakKGReasonerConfig(
    query_parser_config=QueryLLMParserConfig(lang='en'),
    knowledge_comparator_config=KnowledgeComparatorConfig(),
    knowledge_retriever_config=KnowledgeRetrieverConfig(),
    answer_generator_config=QALLMGeneratorConfig(lang='en')
)
pprint(weak_reasoner_config, depth=1, width=200)

WeakKGReasonerConfig(lang='auto',
                     log=<src.utils.logger.Logger object at 0x7f4a295e9390>,
                     verbose=False,
                     query_parser_config=QueryLLMParserConfig(lang='en',
                                                              log=<src.utils.logger.Logger object at 0x7f4a2946a350>,
                                                              verbose=False,
                                                              agent_gen_stategy=None,
                                                              agent_tasks_config=QueryLLMParserAgentTasksConfig(task_to_selector_mapping={...},
                                                                                                                kw_extraction='v2'),
                                                              max_entities=20,
                                                              cache_table_name='qa_queryparser_stage_cache'),
                     knowledge_co

In [13]:
weak_reasoner = WeakKGReasoner(kg_model, weak_reasoner_config)

In [10]:
QUERY_EXAMPLES = ["Which of the following people walked along the highway: Sasha, Masha, Katya?",
                  "Have motorboat was ever sailed through a water canal?",
                  "Have motorboat was ever sailed through a river?"]

In [15]:
for query in QUERY_EXAMPLES:
    print("\nQuery: ", query)
    answer, rinfo, trace = weak_reasoner.perform(query)
    print('Return status: ')
    pprint(rinfo)
    print("Answer: ", answer)


Query:  Which of the following people walked along the highway: Sasha, Masha, Katya?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  Sasha, Masha

Query:  Have motorboat was ever sailed through a water canal?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  <|NotEnoughtInfo|>

Query:  Have motorboat was ever sailed through a river?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  Yes


In [16]:
trace.detailed_result.get_results_sequence()

[(<ModuleType.stage: 'stage'>,
  'extract_entities',
  CompositeModuleSummaryResult(context={'positional_arguments': (QueryInfo(query='Have motorboat was ever sailed through a river?', entities=None, linked_nodes=None, linked_nodes_by_entities=None),)}, result=['motorboat', 'river'], status=<ReturnStatus.success: 0>, elapsed_time=0.54375, cache_hit=False)),
 (<ModuleType.step: 'step'>,
  'match_entities_to_kgnodes',
  SimpleModuleResult(context={'positional_arguments': (QueryInfo(query='Have motorboat was ever sailed through a river?', entities=['motorboat', 'river'], linked_nodes=None, linked_nodes_by_entities=None),)}, result=([NodeInfo(id='668dad0b085703251ab4abcad8789cbd', type=<NodeType.object: 'object'>, text='motorboat'), NodeInfo(id='a5b03048ebe345c488e0ca30eff6ab0c', type=<NodeType.object: 'object'>, text='river')], [['motorboat'], ['river']]), status=<ReturnStatus.success: 0>, elapsed_time=0.0426, cache_hit=False)),
 (<ModuleType.stage: 'stage'>,
  'traverse_knowledge_graph',

In [17]:
print("qa tgen cache:")
pprint(weak_reasoner.get_agent_tgen_stat())
print("qa casual cache:")
pprint(weak_reasoner.get_cache_stat())

qa tgen cache:
{'answer_generator': {'answer_generator_solver': None},
 'query_parser': {'kw_extraction_solver': None}}
qa casual cache:
{'WeakKGReasoner': None,
 'answer_generator': {'QALLMGenerator': None, 'answer_generator_solver': None},
 'knowledge_comparator': {'KnowledgeComparator': None},
 'knowledge_retriever': {'KnowledgeRetriever': None,
                         'triplets_filter': {'TripletsFilter': None},
                         'triplets_retriever': {'MixturedTripletsRetriever': {'MixturedTripletsRetriever': None,
                                                                              'traversal_cache': {'BeamSearchTripletsRetriever': {'BeamSearchTripletsRetriever': None},
                                                                                                  'WaterCirclesRetriever': {'WaterCirclesRetriever': None}}}}},
 'query_parser': {'QueryLLMParser': None, 'kw_extraction_solver': None}}


4.2. Инициализация 'medium' KGReasoner-стадии

In [11]:
medium_reasoner_config = MediumKGReasonerConfig(
    searchplan_enhancer_config=SearchPlanEnhancerConfig(lang='en'),
    entities_extractor_config=EntitiesExtractorConfig(lang='en'),
    e2n_matcher_config=Entities2NodesMatcherConfig(max_n=1),
    cluequeries_generator_config=ClueQueriesGeneratorConfig(lang='en'),
    knowledge_retriever_config=KnowledgeRetrieverConfig(),
    clueanswer_generator_config=ClueAnswerGeneratorConfig(lang='en'),
    clueanswers_summarizer_config=ClueAnswersSummarizerConfig(lang='en'),
    answer_generator_config=AnswerGeneratorConfig(lang='en')
)
pprint(medium_reasoner_config, depth=1, width=200)

MediumKGReasonerConfig(lang='auto',
                       log=<src.utils.logger.Logger object at 0x7f912a4fc850>,
                       verbose=False,
                       searchplan_enhancer_config=SearchPlanEnhancerConfig(lang='en',
                                                                           log=<src.utils.logger.Logger object at 0x7f912a4fc370>,
                                                                           verbose=False,
                                                                           agent_gen_stategy=None,
                                                                           agent_tasks_config=SearchPlanEnhancerAgentTasksConfig(task_to_selector_mapping={...},
                                                                                                                                 plan_initing='v2',
                                                                                                                                 enh

In [12]:
medium_reasoner = MediumKGReasoner(kg_model, medium_reasoner_config)

In [13]:
for query in QUERY_EXAMPLES:
    print("\nQuery: ", query)
    answer, rinfo, trace = medium_reasoner.perform(query)
    print('Return status: ')
    pprint(rinfo)
    print("Answer: ", answer)


Query:  Which of the following people walked along the highway: Sasha, Masha, Katya?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  Sasha and Masha.

Query:  Have motorboat was ever sailed through a water canal?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  <|NotEnoughtInfo|>

Query:  Have motorboat was ever sailed through a river?
Return status: 
ReturnInfo(occurred_warning=[], status=<ReturnStatus.success: 0>, message='')
Answer:  <|NotEnoughtInfo|>


In [14]:
trace.detailed_result.get_results_sequence()

[(<ModuleType.stage: 'stage'>,
  'update_searchplan',
  CompositeModuleSummaryResult(context={'positional_arguments': (0, SearchPlanInfo(base_query='Have motorboat was ever sailed through a river?', search_steps=[], steps_answers=[]))}, result=SearchPlanInfo(base_query='Have motorboat was ever sailed through a river?', search_steps=['Have any motorboats been used for sailing on rivers?', 'Are there any records of motorboats being used to sail through rivers?'], steps_answers=[]), status=<ReturnStatus.success: 0>, elapsed_time=0.96573, cache_hit=False)),
 (<ModuleType.stage: 'stage'>,
  'prepare_searchqueries',
  CompositeModuleSummaryResult(context={'positional_arguments': ('Have any motorboats been used for sailing on rivers?', 0)}, result=[QueryInfo(query='Has Masha been used for sailing on Masha?', entities=['motorboats', 'rivers'], linked_nodes=[NodeInfo(id='c3cc6e312d2bad42cf535aac3a259abd', type=<NodeType.object: 'object'>, text='masha'), NodeInfo(id='c3cc6e312d2bad42cf535aac3a25

In [15]:
print("qa tgen cache:")
pprint(medium_reasoner.get_agent_tgen_stat())
print("qa casual cache:")
pprint(medium_reasoner.get_cache_stat())

qa tgen cache:
{'answer_generator': {'answer_classify_solver': None,
                      'answer_gen_solver': None},
 'clueanswer_generator': {'cagen_solver': None},
 'clueanswers_summarizer': {'clueanswers_summ_solver': None},
 'cluequeries_generator': {'cluequery_gen_solver': None},
 'entities_extractor': {'entities_extractor_solver': None},
 'searchplan_enhancer': {'enhance_classify_solver': None,
                         'plan_enhancing_solver': None,
                         'plan_initialing_solver': None}}
qa casual cache:
{'MediumKGReasoner': None,
 'answer_generator': {'AnswerGenerator': None,
                      'answer_classify_solver': None,
                      'answer_gen_solver': None},
 'clueanswer_generator': {'ClueAnswerGenerator': None, 'cagen_solver': None},
 'clueanswers_summarizer': {'ClueAnswersSummarizer': None,
                            'clueanswers_summ_solver': None},
 'cluequeries_generator': {'ClueQueriesGenerator': None,
                           'c

In [ ]:
kg_model.close_connections()
mem_pipeline.close_connections()
del kg_model
del mem_pipeline

: 